# R1-03 · Cruzar y resumir tablas — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R1 · *Analista de Datos* · Semana 4 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Identificar la **llave** que une dos tablas.
2. Cruzar con **pd.merge** y entender qué produce.
3. Distinguir **inner vs left** y cazar filas huérfanas.
4. Responder la pregunta real con **groupby** tras el cruce.

**Competencia de salida:** cruzar tablas del Estado con criterio: el BUSCARV profesional, con auditoría de lo que no calza.

**Dato real:** matrícula 2025 por RBD × directorio oficial de establecimientos (MINEDUC) — con 4 huérfanas reales.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd

# "En vivo o caché": usa los archivos locales; si faltan (ej. Colab), se bajan del repo.
for archivo in ["matricula_rbd.csv", "directorio_establecimientos.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)

df_matricula = pd.read_csv("matricula_rbd.csv")          # conteos de estudiantes por RBD
df_directorio = pd.read_csv("directorio_establecimientos.csv")  # quién es cada RBD
print("matrícula:", df_matricula.shape, "| directorio:", df_directorio.shape)
df_matricula.head(3)

## ¿Por qué dos tablas? La llave que las une

El MINEDUC no guarda el nombre, la comuna y la dependencia del establecimiento **en cada
registro de matrícula**: eso duplicaría la misma información miles de veces. En su lugar,
cada establecimiento tiene un código único — el **RBD** (Rol Base de Datos) — y hay dos tablas:

- `df_matricula`: **cuántos** estudiantes tiene cada RBD (la medida).
- `df_directorio`: **quién es** cada RBD — nombre, dependencia, región, comuna (el catálogo).

Para responder «¿qué región tiene más estudiantes?» necesitas las dos. El **cruce** (`merge`)
es el BUSCARV profesional: le pega a cada fila de matrícula los atributos de su establecimiento.

> Dato real: las dos tablas NO calzan perfecto — hay 4 RBD que reportan matrícula pero no
> figuran «funcionando» en el directorio oficial. Los vas a cazar en el ejercicio 3.

## 1. Encontrar la llave

Antes de cruzar hay que identificar la **llave**: la columna que existe en ambas tablas y significa lo mismo. Mira `df_matricula.columns` y `df_directorio.columns`.

### ✍️ Ejercicio 1 — Identifica la llave y el tamaño de cada tabla

In [ ]:
# TODO: nombre de la columna que está en AMBAS tablas (entre comillas)
llave = ...
# TODO: filas de cada tabla
n_matricula = ...
n_directorio = ...
print("llave:", llave, "| matrícula:", n_matricula, "| directorio:", n_directorio)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert llave in df_matricula.columns and llave in df_directorio.columns
    assert n_matricula == len(df_matricula)
    assert n_directorio == len(df_directorio)
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "Compara las listas de columnas: la común es 'rbd'. len(...) para los tamaños.")
    print("   Detalle:", e)

## 2. El primer cruce: `pd.merge`

`pd.merge(izquierda, derecha, on=llave)` pega las columnas de la derecha a cada fila de la izquierda cuando la llave coincide. Cruza matrícula con directorio.

### ✍️ Ejercicio 2 — Cruza las dos tablas

In [ ]:
# TODO: cruza df_matricula con df_directorio por la llave "rbd"
df_cruce = ...
n_cruce = ...
print("filas del cruce:", n_cruce)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert "region" in df_cruce.columns and "mat_total" in df_cruce.columns
    assert n_cruce == len(pd.merge(df_matricula, df_directorio, on="rbd"))
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'pd.merge(df_matricula, df_directorio, on="rbd") y len(...).')
    print("   Detalle:", e)

## 3. Inner vs left: cazar la fila huérfana

El cruce anterior (inner, el modo por defecto) **botó** los RBD que no están en el directorio. Con `how="left"` conservas TODAS las filas de matrícula; las que no encontraron pareja quedan con la región **nula** — y las puedes contar.

### ✍️ Ejercicio 3 — Encuentra los establecimientos huérfanos

In [ ]:
# TODO: cruce que conserva TODAS las filas de matrícula (how="left")
df_left = ...
# TODO: cuántas quedaron sin datos de directorio (region nula)
n_huerfanas = ...
print("filas totales:", len(df_left), "| huérfanas (sin directorio):", n_huerfanas)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert len(df_left) == len(df_matricula)
    assert n_huerfanas == int(df_left["region"].isna().sum())
    assert n_huerfanas > 0
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'pd.merge(..., how="left") y df_left["region"].isna().sum().')
    print("   Detalle:", e)

## 4. La pregunta real: ¿qué región concentra los estudiantes?

Cruzar es el medio; el fin es **responder algo**. Ahora que cada conteo tiene su región, `groupby` responde la pregunta de gestión.

### ✍️ Ejercicio 4 — Estudiantes por región y la región líder

In [ ]:
# TODO: estudiantes (mat_total) por región, de mayor a menor
est_por_region = ...
# TODO: nombre de la región con más estudiantes
region_top = ...
print("región con más estudiantes:", region_top)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert len(est_por_region) == df_cruce["region"].nunique()
    assert region_top == est_por_region.idxmax()
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'groupby("region")["mat_total"].sum().sort_values(ascending=False); el líder es .idxmax().')
    print("   Detalle:", e)

## ✍️ Reflexión (no se corrige, pero es lo más importante)

Los 4 RBD huérfanos que cazaste en el ejercicio 3 reportan estudiantes pero no figuran
«funcionando» en el registro oficial. En una o dos frases: ¿qué harías con esas filas en un
informe — las botas, las incluyes, las reportas aparte? ¿A quién le preguntarías por qué existen?

> Esa pregunta — qué hacer con los datos que no calzan — es el trabajo real del analista.

## Errores típicos al cruzar

- **`KeyError` en la llave** → no se llama igual en ambas tablas; usa `left_on=`/`right_on=` si difieren.
- **El cruce tiene MÁS filas que las tablas** → la llave está repetida en alguna tabla (relación 1-a-muchos).
- **Todo quedó nulo** → cruzaste por la columna equivocada o los tipos difieren (texto vs número).

## Cierre

- La **llave** conecta la medida con el catálogo; el **RBD** es la cédula de identidad de cada escuela.
- `inner` bota lo que no calza; `left` lo conserva como **nulo** — y contarlos es auditar.
- El cruce es el medio: la pregunta de gestión se responde con **groupby** después de cruzar.

> *En R1-04 aprenderás a limpiar datos sucios — porque las llaves que no calzan y los valores
> imposibles son el pan de cada día.*